In [1]:
! pip install -q transformers datasets accelerate evaluate scikit-learn huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from transformers import AutoModelForSequenceClassification
import math

In [3]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False No GPU


In [4]:
model_checkpoint = "distilbert-base-uncased"
base_model = AutoModel.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
print(base_model)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [6]:
id2label = {0: "negative", 1:"neutral", 2:"positive"}
label2id = {"negative":0, "neutral":1, "positive":2}

classification_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
     num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
print(classification_model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [8]:
print(classification_model.config)

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "vocab_size": 30522
}



In [ ]:
# Full list of every parameter name + its shape, in the base encoder (no head)
# for name, params in base_model.state_dict().items():
#   print(f"{name}  ------>       {tuple(params.shape)}")

In [9]:
all_keys = list(base_model.state_dict().keys())
print(f"Total number of parameter tensors: {len(all_keys)}")

# Should be: 4 (embeddings) + 6 layers x 16 params/layer = 4 + 96 = 100

Total number of parameter tensors: 100


In [29]:

class DistilBertEmbeddings(nn.Module):

    def __init__(self, vocab_size=30522, hidden_size=768, max_position_embeddings=512, dropout_prob=0.1):
        super().__init__()

        self.word_embeddings = nn.Embedding(vocab_size, hidden_size)
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        self.LayerNorm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        token_embeds = self.word_embeddings(input_ids)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        position_embeds = self.position_embeddings(position_ids)

        embeddings = token_embeds + position_embeds
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)

        return embeddings

In [30]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, dropout_prob=0.1):
        super().__init__()

        assert hidden_size % num_heads == 0 #hidden_size must be divisible by num_heads

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads  # 768 // 12 = 64

        self.q_lin = nn.Linear(hidden_size, hidden_size)
        self.k_lin = nn.Linear(hidden_size, hidden_size)
        self.v_lin = nn.Linear(hidden_size, hidden_size)

        self.out_lin = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout_prob)

    def split_heads(self, x, batch_size):
        seq_len = x.shape[1]
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)
        return x

    def forward(self, hidden_states, attention_mask):
        batch_size = hidden_states.shape[0]

        Q = self.q_lin(hidden_states)
        K = self.k_lin(hidden_states)
        V = self.v_lin(hidden_states)

        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        extended_mask = attention_mask[:, None, None, :]
        scores = scores.masked_fill(extended_mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = torch.matmul(attention_weights, V)

        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, -1, self.hidden_size)

        output = self.out_lin(context)

        return output

In [31]:
class FeedForward(nn.Module):

    def __init__(self, hidden_size=768, intermediate_size=3072, dropout_prob=0.1):
        super().__init__()

        self.lin1 = nn.Linear(hidden_size, intermediate_size)
        self.activation = nn.GELU()
        self.lin2 = nn.Linear(intermediate_size, hidden_size)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        x = self.lin1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.lin2(x)
        return x


In [32]:
class TransformerBlock(nn.Module):
  def __init__(self, hidden_size=768, num_heads=12, intermediate_size=3072, dropout_prob=0.1):
    super().__init__()
    self.attention = MultiHeadSelfAttention(
        hidden_size=hidden_size,
        num_heads=num_heads,
        dropout_prob=dropout_prob
    )
    self.ffn = FeedForward(
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        dropout_prob=dropout_prob
    )
    self.sa_layer_norm = nn.LayerNorm(hidden_size, eps=1e-12, elementwise_affine=True)
    self.output_layer_norm = nn.LayerNorm(hidden_size, eps=1e-12, elementwise_affine=True)
    # self.dropout = nn.Dropout(dropout_prob)

  def forward(self, input_embeddings, attention_mask):
    attention_output = self.attention(input_embeddings, attention_mask)
    # print(f"attention layer output shape: {attention_output.shape}")

    attention_output_add_and_norm = self.sa_layer_norm(attention_output+input_embeddings)
    ffn_output = self.ffn(attention_output_add_and_norm)
    transformer_block_output = self.output_layer_norm(ffn_output+attention_output_add_and_norm)

    return transformer_block_output


In [33]:
class TransformerEncoder(nn.Module):
  def __init__(self, hidden_size=768, num_heads=12, intermediate_size=3072,
               num_layers=6, dropout_prob=0.1):
    super().__init__()
    self.layer = nn.ModuleList([
      TransformerBlock(
          hidden_size=hidden_size,
          num_heads=num_heads,
          intermediate_size=intermediate_size,
          dropout_prob=dropout_prob
      )
      for _ in range(num_layers)
    ])
  def forward(self, inputs, attention_mask):
    hidden_state = inputs
    for each_layer in self.layer:
      hidden_state = each_layer(hidden_state, attention_mask)
    return hidden_state

In [34]:
class DistilBertModel(nn.Module):

    def __init__(self, vocab_size=30522, hidden_size=768, num_layers=6,
                 num_heads=12, intermediate_size=3072,
                 max_position_embeddings=512, dropout_prob=0.1):
        super().__init__()

        self.embeddings = DistilBertEmbeddings(
            vocab_size=vocab_size,
            hidden_size=hidden_size,
            max_position_embeddings=max_position_embeddings,
            dropout_prob=dropout_prob
        )

        self.transformer = TransformerEncoder(
    hidden_size=hidden_size,
    num_heads=num_heads,
    intermediate_size=intermediate_size,
    num_layers=num_layers,
    dropout_prob=dropout_prob
            )

    def forward(self, input_ids, attention_mask):
        embeddings = self.embeddings(input_ids)
        hidden_states = self.transformer(embeddings, attention_mask)
        return hidden_states

In [35]:
class DistilBertForSequenceClassification(nn.Module):
  def __init__(self, vocab_size=30522, hidden_size=768, num_layers=6,
                 num_heads=12, intermediate_size=3072,
                 max_position_embeddings=512, dropout_prob=0.1,):
    super().__init__()

    self.distilbert = DistilBertModel(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_heads=num_heads,
    intermediate_size=intermediate_size,
    max_position_embeddings=max_position_embeddings,
    dropout_prob=dropout_prob
                              )

    self.pre_classifier = nn.Linear(in_features=hidden_size, out_features=hidden_size)
    self.classifier = nn.Linear(in_features=hidden_size, out_features=3)
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, input_ids, attention_mask):
    attention_output = self.distilbert(input_ids, attention_mask)

    pooled_output = attention_output[:,0,:]
    pooled_output = self.pre_classifier(pooled_output)
    pooled_output = self.dropout(pooled_output)
    logits = self.classifier(pooled_output)

    return logits


In [36]:
# with classifier head testing
input_ids = tokenizer(["I hat this", "I love this!"], padding="max_length", max_length=128, return_tensors="pt")["input_ids"]
attention_mask = tokenizer(["I hat this", "I love this!"], padding="max_length", max_length=128, return_tensors="pt")["attention_mask"]

test_model = DistilBertForSequenceClassification()
logits = test_model(input_ids, attention_mask)

print(logits.shape)
probs = torch.softmax(logits, dim=-1)


torch.Size([2, 3])


In [37]:
#loading pretrained model weights into the custom model layers
Farhan_model = DistilBertForSequenceClassification()
Farhan_model.distilbert.load_state_dict(base_model.state_dict())

<All keys matched successfully>

In [39]:
sd_custom = Farhan_model.distilbert.state_dict()
sd_base = base_model.state_dict()

mismatches = []
for name, tensor in sd_base.items():
    if name not in sd_custom:
        mismatches.append(f"MISSING in custom model: {name}")
    elif not torch.equal(sd_custom[name], tensor):
        mismatches.append(f"VALUE MISMATCH: {name}")

if not mismatches:
    print(f"All {len(sd_base)} tensors verified identical.")
else:
    for m in mismatches:
        print(m)

All 100 tensors verified identical.


In [40]:
Farhan_model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): DistilBertEmbeddings(
      (word_embeddings): Embedding(30522, 768)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): TransformerEncoder(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (ffn): FeedForward(
            (lin1): Linear(in_features=768, out_features=3072, bias=True)
            (activation): GELU(approximate='none')


In [59]:
list(Farhan_model.state_dict().items())[1][:]

('distilbert.embeddings.position_embeddings.weight',
 tensor([[ 1.7505e-02, -2.5631e-02, -3.6642e-02,  ...,  3.3437e-05,
           6.8312e-04,  1.5441e-02],
         [ 7.7580e-03,  2.2613e-03, -1.9444e-02,  ...,  2.8910e-02,
           2.9753e-02, -5.3247e-03],
         [-1.1287e-02, -1.9644e-03, -1.1573e-02,  ...,  1.4908e-02,
           1.8741e-02, -7.3140e-03],
         ...,
         [ 1.7418e-02,  3.4903e-03, -9.5621e-03,  ...,  2.9599e-03,
           4.3435e-04, -2.6949e-02],
         [ 2.1687e-02, -6.0216e-03,  1.4736e-02,  ..., -5.6118e-03,
          -1.2590e-02, -2.8085e-02],
         [ 2.6413e-03, -2.3298e-02,  5.4922e-03,  ...,  1.7537e-02,
           2.7550e-02, -7.7656e-02]]))

In [58]:
list(base_model.state_dict().items())[1][:20]

('embeddings.position_embeddings.weight',
 tensor([[ 1.7505e-02, -2.5631e-02, -3.6642e-02,  ...,  3.3437e-05,
           6.8312e-04,  1.5441e-02],
         [ 7.7580e-03,  2.2613e-03, -1.9444e-02,  ...,  2.8910e-02,
           2.9753e-02, -5.3247e-03],
         [-1.1287e-02, -1.9644e-03, -1.1573e-02,  ...,  1.4908e-02,
           1.8741e-02, -7.3140e-03],
         ...,
         [ 1.7418e-02,  3.4903e-03, -9.5621e-03,  ...,  2.9599e-03,
           4.3435e-04, -2.6949e-02],
         [ 2.1687e-02, -6.0216e-03,  1.4736e-02,  ..., -5.6118e-03,
          -1.2590e-02, -2.8085e-02],
         [ 2.6413e-03, -2.3298e-02,  5.4922e-03,  ...,  1.7537e-02,
           2.7550e-02, -7.7656e-02]]))